In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

class_names = [
    "Glioma",
    "Meningioma",
    "No Tumor",
    "Pituitary"
]

In [4]:
val_df = pd.read_csv("../data/processed/val.csv")

print("Validation images:", len(val_df))
print(val_df.head())

Validation images: 512
  split   class                                       image  \
0   Val  Glioma   ../data/raw/Val/Glioma/images/gg (10).jpg   
1   Val  Glioma  ../data/raw/Val/Glioma/images/gg (131).jpg   
2   Val  Glioma  ../data/raw/Val/Glioma/images/gg (132).jpg   
3   Val  Glioma  ../data/raw/Val/Glioma/images/gg (135).jpg   
4   Val  Glioma  ../data/raw/Val/Glioma/images/gg (136).jpg   

                                   annotation  label  
0   ../data/raw/Val/Glioma/labels/gg (10).txt      0  
1  ../data/raw/Val/Glioma/labels/gg (131).txt      0  
2  ../data/raw/Val/Glioma/labels/gg (132).txt      0  
3  ../data/raw/Val/Glioma/labels/gg (135).txt      0  
4  ../data/raw/Val/Glioma/labels/gg (136).txt      0  


In [5]:
val_df["image"] = val_df["image"].str.replace(
    "/raw/val/",
    "/raw/Val/",
    regex=False
)

missing = sum(
    not Path(p).exists()
    for p in val_df["image"]
)

print("Missing images:", missing)

Missing images: 0


In [6]:
def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)
    image.set_shape((224, 224, 3))

    return image, label

In [8]:
val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image"].tolist(),
        val_df["label"].tolist()
    )
)

val_ds = val_ds.map(load_image)
val_ds = val_ds.batch(BATCH_SIZE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

y_true = val_df["label"].to_numpy()

print("Validation dataset ready!")

Validation dataset ready!
